In [0]:
# ===============================================================
# 1. PARÁMETROS DE ENTRADA (NO MODIFICAR)
# Valores inyectados por el orquestador (Databricks Jobs / Bundle)
# ===============================================================

import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
import json
import sys

# --- Widgets de Databricks para ejecución orquestada ---
dbutils.widgets.text("periodo", "202604", "Periodo")
dbutils.widgets.text("mode_environment", "dev", "Mode Environment")

periodo = dbutils.widgets.get("periodo")
mode_environment = dbutils.widgets.get("mode_environment")

ENV = mode_environment
MES_VTA = int(periodo)

print(f"Ejecutando notebook para el periodo: {MES_VTA} en entorno: {ENV}")


In [0]:
#====================================================
# 2. CONFIGURACIÓN EDITABLE - MLE
# Variables principales que deben ser adaptadas para cada nuevo modelo o caso de uso.
#====================================================

# --- 2.1 Workspace y MLFlow ---
#WORKSPACE = "/Workspace/Users/natalivconcha@bcp.com.pe/cemm-dtbr-deploy-score-yape/src"
WORKSPACE = "/Workspace/Users/marioavento@bcp.com.pe/cemm-202605-dtr-deploy-troncal-bhv-pyme/src"
REGISTERED_MODEL_NAME = "catalog_cemm_expl_bcp_prod.bcp_expl_007_models.catboost_pd_bhv_pyme"

# --- 2.2 Metadatos del Modelo ---
USE_CASE = "pd_bhv_pyme"
CAMPO_PRIMARIO = "codclaveunicocli"
SCORE_NAME_MODEL = "PD"                # Nombre de la columna de score output del predict

# --- 2.3 Nombres Base de Tablas ---
SCORING_TABLE_SAS = "catalog_cemm_expl_bcp.prod.bcp_expl_007." #Tabla scoreada
SOURCE_TABLE = f"catalog_lhcl_prod_bcp.bcp.ddv_adrmmgr_seginfobasesgenerales_vu.hm_portafoliocredito"
#BASE_NAME_TABLE_MASTER = "bcp_mt_troncal_yape"
BASE_NAME_TABLE_MASTER = "bcp_mt_troncal_bhv_pyme_v1"
BASE_NAME_TABLE_OUTPUT = "bcp_tb_score_pd_bhv_pyme_v1"

MODEL_NAME_PYTHON = "model_v2_REDUCC_32V.pkl"

# --- 2.4 Comportamiento de Ejecución y Debugging ---
FIRST_RUN = True                     # Flag de primer run para cálculos de calidad
MESES_HISTORICOS_CALIDAD = 1         # Rango en meses para validación histórica de datos
DEBUG_MLOPS = False                  # Activar limite de muestras (Debug logs)
DEBUG_ROWS = 1000                    # Cantidad limite de filas procesadas si Debug es True

# --- 2.5 Certificación ---
CRITICAL_FIELDS = [
    'numpd'
]

JOIN_KEYS = ['codclaveunicol']
EXCLUDE_COLUMNS = []
CERTIFICATION_THRESHOLDS = {
    'records': {'green': 99.0},
    'schema': {'green': 100.0},
    'data_match': {'green': 99.0, 'yellow': 96.0},
    'critical_fields': {'green': True}
}

# --- 2.6 Configuracion LHCL ---
LHCL_CATALOG = ""
LHCL_SCHEMA = ""
LHCL_TABLE = ""

# Columnas fijas (nivel raíz): nombre_destino -> nombre_origen
LHCL_FIXED_COLUMNS = {
    "codclaveunicocli": "codclaveunicocli",
    "codclavepartycli": "codclavepartycli",
    "numpd": "numpd",
    "codmes":"codmes",
}

# Tipo destino LHCL para cada campo del struct colvariablesmodeloanalitico
LHCL_STRUCT_TYPES = {

}

# --- 2.7 Lista de Features del modelo ----

FEATURE_NAMES = [
    'MTX_RCC_OTRA_DEUDA__rcc_mto_rdv_max_u3m_cap',
    'CLASI_EXPER_CLI__ctdempleado_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_pas_max_min_12_12_rt_u12_cap',
    'EVOL_CLI_PYM__ctdmaxdiamorau6m_cap',
    'MTX_RESUMEN_SALDO__prod_ctd_sld_act_u1m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_act_min_6_6_rt_u6m',
    'MOD_DEMO__ctdmesantiguedadempsunat_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_med_1_6_rt_u6m',
    'APP_SCORE_APROB_PYME__utl_3_rl_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24_cap',
    'APP_SCORE_APROB_PYME__montoade_act_max6_s_hip_rl',
    'MTX_TRX_CANAL__can_tkt_tmo_tot_sol_min_u12_cap',
    'MTX_RESUMEN_ACT_PAS__prod_pct_pmtsav_pmact_24_24_rt_u24_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_med_1_6_rt_u6m_cap',
    'MOD_ACT__pctratiomtodecdeudapymertmtopasivoprmu3m_cap',
    'MTX_MOV_CARGO_PAS__isav_tkt_opec_pago_srv_prm_u3m',
    'APP_SCORE_APROB_PYME__edad_fin_cap',
    'PASIVO_EVOL_SALD_PYM__mtoprmincrvariacionmensualprmvigsolu6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_act_max_min_6_6_rt_u6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_12_12_rt_u12',
    'MTX_RCC_PROD__rcc_tip_cond_mor_max_crnor_max_u6m_cap',
    'VIDEVAR_MTX_MORA_POND_CLI_MMGR__mtodeudaclasifriesgofactordsctosolu12_cap',
    'RNG_ACTIVIDAD_ECONOM',
    'MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_min_12_12_rt_u12_cap',
    'MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_dol_max_u3m_cap',
    'MOD_ACT__pctratiomtoopeaprmu6mopecprmu12_cap',
    'MTX_TRX_CANAL_PAGO_TRANSF__can_mto_tmo_tot_pag_bcp_prm_u6m_cap',
    'APP_SCORE_APROB_PYME__meses_activo_sf_bu_ma6_0_ag',
    'MTX_TRX_CANAL_PAGO_TRANSF__can_ctd_tmo_tot_pag_bcp_frq_u6m',
    'APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl_cap',
]

# --- 2.8 Control de versionado de tablas ---
TABLE_VERSION_SUFFIX = "" # Para producción o primera versión (vacío)
# TABLE_VERSION_SUFFIX = "_v2" # Descomentar cuando se quiera generar v2

In [0]:
#====================================================
# 3. CONFIGURACIÓN COMPLETA DEL SISTEMA MLOPS (NO MODIFICAR RUTINARIAMENTE)
# Resolución de Storage, schemas y catálogos basados en el environment
#====================================================

ENVIRONMENT_CONFIG = {
    "dev": {
        "CONTAINER": "bcp-expl-007",
        "STORAGE_ACCOUNT": "adlscu1cemmbackp07",
        "CATALOG_NAME": "catalog_cemm_expl_bcp_prod",
        "SCHEMA_NAME": "bcp_expl_007",
        "SCHEMA_MODELS": "bcp_expl_007_models",
    },
    "stg": {
        "CONTAINER": "bcp-expl-007",
        "STORAGE_ACCOUNT": "adlscu1cemmbackp07",
        "CATALOG_NAME": "catalog_cemm_expl_bcp_prod",
        "SCHEMA_NAME": "bcp_expl_007",
        "SCHEMA_MODELS": "bcp_expl_007_models",
    },
    "prod": {
        "CONTAINER": "bcp-expl-008",
        "STORAGE_ACCOUNT": "adlscu1cemmbackp08",
        "CATALOG_NAME": "catalog_cemm_desp_bcp_prod",
        "SCHEMA_NAME": "bcp_expl_008",
        "SCHEMA_MODELS": "bcp_expl_008_models",
    },
}

if ENV not in ENVIRONMENT_CONFIG:
    raise ValueError(f"Ambiente '{ENV}' no válido. Use 'dev', 'stg' o 'prod'")

_env_cfg = ENVIRONMENT_CONFIG[ENV]
CONTAINER = _env_cfg["CONTAINER"]
STORAGE_ACCOUNT = _env_cfg["STORAGE_ACCOUNT"]
CATALOG_NAME = _env_cfg["CATALOG_NAME"]
SCHEMA_NAME = _env_cfg["SCHEMA_NAME"]
SCHEMA_MODELS = _env_cfg["SCHEMA_MODELS"]

print(f"Configuración de ambiente '{ENV}' cargada: Catalog={CATALOG_NAME}, Schema={SCHEMA_NAME}")

# Construcción de nombres finales de tablas orquestadas
#TABLE_MASTER = f"{CATALOG_NAME}.{SCHEMA_NAME}.{BASE_NAME_TABLE_MASTER}_{env}"
TABLE_MASTER = f"{CATALOG_NAME}.{SCHEMA_NAME}.{BASE_NAME_TABLE_MASTER}"
TABLE_OUTPUT = f"{BASE_NAME_TABLE_OUTPUT}_{ENV}"

In [0]:
#====================================================
# 4. CONFIGURACIÓN AUTOMÁTICA - VARIABLES DERIVADAS (NO MODIFICAR)
#====================================================

# --- Configuración de Storage (derivada) ---
_work_path = f"{ENV}/bcp/ddv/mlops/mmgr"
ADLS_LOCATION_TABLE = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{_work_path}"

# --- Tabla de scoring output completa ---
OUTPUT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_OUTPUT}"

# --- Project Path ---
try:
    PROJECT_PATH = '/Workspace' + os.path.dirname(
        os.path.dirname(
            dbutils.notebook.entry_point.getDbutils().
            notebook().getContext().notebookPath().get()
        )
    )
except Exception:
    PROJECT_PATH = WORKSPACE

sys.path.insert(0, str(PROJECT_PATH))
from utils.month_delta import month_delta

# --- Paths del Modelo ---
MODEL_FILE_PATH = f"{PROJECT_PATH}/data/models/{MODEL_NAME_PYTHON}"

# ---- Paths Locales (Workspace) ---
LOGS_PATH = f"{PROJECT_PATH}/logs"
QUALITY_CONFIG = f"{PROJECT_PATH}/config/quality_config"
TEMPLATE_REPORT_PATH = f"{PROJECT_PATH}/data/reports/template"
MAPPING_PATH = f"{PROJECT_PATH}/data/mapping"

# --- Paths Temporales de Reportes ---
TEMP_REPORTS_OUTPUT_PATH = f"{PROJECT_PATH}/data/reports/tmp/output/{MES_VTA}"
TEMP_REPORTS_INPUT_PATH = f"{PROJECT_PATH}/data/reports/tmp/input/{MES_VTA}"
TEMP_REPORTS_CERTIFICATION_PATH = f"{PROJECT_PATH}/data/reports/tmp/certification/{MES_VTA}"

# --- Configuracion LHCL (derivada) ---
LHCL_OUTPUT_TABLE = f"{LHCL_CATALOG}.{LHCL_SCHEMA}.{LHCL_TABLE}"

print("-"*90)
print("✓ Configuración cargada exitosamente")
print("-"*90)

In [0]:
#====================================================
# 5. CONFIGURACIÓN DE LOGGING
#====================================================

from utils.logger import MLOpsLogger

# Crear directorio de logs si no existe
os.makedirs(LOGS_PATH, exist_ok=True)

# Inicializar timestamp del job (compartido entre todos los notebooks)
# Cada notebook creará su propio logger, pero todos escribirán con el mismo prefijo
_init_logger = MLOpsLogger(
    name='config_init',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    enable_console=False,
    enable_file=True,
    enable_json=True,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'use_case': USE_CASE,
        'debug_mode': DEBUG_MLOPS
    }
)

JOB_TIMESTAMP = MLOpsLogger.get_current_job_prefix()
_init_logger.close()

print(f"Job Timestamp: {JOB_TIMESTAMP}")
print(f" Archivos de log: job_{JOB_TIMESTAMP}_*.log")

In [0]:
#===================================================
# 6. RESUMEN DE CONFIGURACIÓN
#====================================================

print("-"*90)
print("RESUMEN DE CONFIGURACIÓN")
print("-"*90)

print(f"\n EJECUCIÓN:")
print(f"  Periodo: {MES_VTA}")
print(f"  Ambiente: {ENV}")
print(f"  Debug Mode: {DEBUG_MLOPS}")
if DEBUG_MLOPS:
    print(f"  Debug Rows: {DEBUG_ROWS}")

print(f"\n CATÁLOGO:")
print(f"  Catalog: {CATALOG_NAME}")
print(f"  Schema: {SCHEMA_NAME}")

print(f"\n TABLAS:")
print(f"  Master Table: {TABLE_MASTER}")
print(f"  Output Table: {OUTPUT_TABLE}")
print(f"  Source Table: {SOURCE_TABLE}")

print(f"\n STORAGE:")
print(f"  ADLS Base: {ADLS_LOCATION_TABLE}")

print(f"\n PATHS:")
print(f"  Project: {PROJECT_PATH}")
print(f"  Logs: {LOGS_PATH}")
print(f"  Model: {MODEL_FILE_PATH}")

print(f"\n MODELO:")
print(f" Use Case: {USE_CASE}")
print(f" Campo Primario: {CAMPO_PRIMARIO}")
print(f" Score Column: {SCORE_NAME_MODEL}")
print(f" Model File: {MODEL_NAME_PYTHON}")

print("\n" + "-"*90)
print("✓ Todas las variables están listas para usar")
print("="*90)

In [0]:
#!pip install --upgrade pip && pip install -r /Workspace/Users/natalivconcha@bcp.com.pe/cemm-dtbr-deploy-score-yape/src/requirements.txt
!pip install -r //Workspace/Users/marioavento@bcp.com.pe/cemm-202605-dtr-deploy-troncal-bhv-pyme/src/requirements.txt
# /Workspace/Users/natalivconcha@bcp.com.pe/202509-cemm-dtr-dataprep-test-batch/src/requirements.txt

In [0]:
COMPARE_COLUMNS = []